<a href="https://colab.research.google.com/github/walter-04/TFM_Canine/blob/feature%2FVibe_codec/notebooks/VIBE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libs

In [1]:
from nibabel.affines import rescale_affine, voxel_sizes
from nibabel.processing import resample_from_to
from scipy.ndimage import affine_transform
from skimage.segmentation import slic, mark_boundaries
from skimage.exposure import match_histograms
from skimage.segmentation import felzenszwalb
from skimage.transform import SimilarityTransform, warp
from skimage.filters import threshold_otsu
from skimage.morphology import disk, dilation
import skimage
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as pltImage
import math
import scipy
import glob
import cv2

In [ ]:
!pip install dipy

In [ ]:
from dipy.align.transforms import AffineTransform2D, RigidTransform2D
from dipy.align.imaffine import (AffineMap,
                                 MutualInformationMetric,
                                 AffineRegistration)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Clase VIBE

> Contiene toda la lógica del proceso VIBE para la segmentación



In [ ]:
class ViBE:

  def __init__(self, atlas, path_out):
    '''
    params:
      atlas: nibabel image object
    '''
    self.atlas = self.reoriented_data(nib.load(atlas),
                                      y=-1,
                                      z=-1)
    self.slide_atlas = 120
    self.path_output = 'path_out'
    #self.point_reference = np.array([100, 150])
    self.point_reference = np.array([120, 175])

  def reoriented_data(self, file_nii, x=1, y=1, z=1):
    '''
    params:
      file_nii: nibabel image object
      x: int
      y: int
      z: int
    '''
    ornt = np.array([[0, x],
                  [1, y],
                  [2, z]])
    return file_nii.as_reoriented(ornt)


  def load_data(self, file_nii):
    '''
    params:
      file_nii: nibabel image object
    '''
    #atlas = reoriented_data(nib.load(atlas), y=-1, z=-1)
    #data_atlas = atlas.get_fdata()
    img_work = self.reoriented_data(nib.load(file_nii), z=-1)
    data_img_work = img_work.get_fdata()
    return img_work, data_img_work

  def normalize_img(self, img):
    '''
    params:
      img: numpy array
    '''
    norm_image = cv2.normalize(img, None, alpha = 0,
                               beta = 255, norm_type = cv2.NORM_MINMAX,
                               dtype = cv2.CV_32F)
    return norm_image


  def prepare_data(self, img_work):
    '''
    params:
      img_work: nibabel image object
    '''
    new_affine_atlas = rescale_affine(self.atlas.affine,
                                shape=self.atlas.shape,
                                zooms=voxel_sizes(img_work.affine),
                                new_shape=img_work.shape)
    atlas_resize = resample_from_to(self.atlas,
                                (img_work.shape, new_affine_atlas),
                                order=1,
                                mode='nearest')
    #self.atlas_resize_data = self.atlas_resize.get_fdata()
    matched_atlas_resize = match_histograms(
                                      atlas_resize.get_fdata(),
                                      img_work.get_fdata())

    matched_atlas_resize =  atlas_resize.get_fdata()


    my_otsu_threshold = threshold_otsu(matched_atlas_resize[self.slide_atlas, :,:])
    mask_atlas = matched_atlas_resize[self.slide_atlas, :,:] > my_otsu_threshold
    mask_atlas = dilation(mask_atlas, disk(10))
    #
    return mask_atlas, matched_atlas_resize


  def center_brain(self, data_img_work, matched_atlas_resize, mask_atlas):
    '''
    params:
      img_nii: numpy array
      matched_atlas_resize: numpy array
      mask_atlas: numpy array
    '''
    #data_img_work_slice = img_nii.get_fdata()[self.slide_img, :, :]
    #data_img_work = img_nii.get_fdata()
    data_img_work_slice = data_img_work[int(data_img_work.shape[0]/2), :, :]

    num_no_background = np.sum(mask_atlas.astype(int) == 1)

    '''
    segments = felzenszwalb(data_img_work_slice,
                        sigma=0.5,
                        scale=200*num_no_background,
                        min_size=100
                        )
    '''

    segments = felzenszwalb(data_img_work_slice,
                        sigma=0.5,
                        scale=200*332,
                        min_size=332)


    centers_segments = np.array([np.mean(np.nonzero(segments==i), axis=1)
                    for i in np.unique(segments)])

    points_distance = [(math.sqrt(
                                  (i[0]-self.point_reference[0])**2 +
                                  (i[1]-self.point_reference[1])**2), i)
                      for i in centers_segments]

    center_optime = np.array(min(points_distance)[1])

    ###
    # este slide --> matched_imagen_atlas_modif[131, :,:]
    atlas_slide_matched = self.normalize_img(matched_atlas_resize[self.slide_atlas, :,:])

    # Buscamos el elemento en el atlas
    segments_atlas = felzenszwalb(atlas_slide_matched,
                        sigma=0.5,
                        scale=200*332,
                        min_size=332*10)

    # Indice de los puntos del atlas-segmentados
    index_brain_atlas = np.transpose(np.nonzero(segments_atlas==1))
    # Centro del volumen
    centers_atlas_brain = np.array([np.mean(np.nonzero(segments_atlas==i),axis=1)
                        for i in np.unique(segments_atlas) if i == 1])

    #Se calcula cuantos pixeles hay que mover  de un sitio a otro
    mov_info = np.absolute(center_optime - centers_atlas_brain).flatten().tolist()


    tform = SimilarityTransform(translation=(mov_info[0], mov_info[1]))
    atlas_slide_moved = affine_transform(atlas_slide_matched,
                                            tform, mode='nearest')

    atlas_mask_moved = affine_transform(mask_atlas,
                                        tform, mode='nearest')

    return atlas_slide_moved, atlas_mask_moved


  def apply_register(self, data_img_work, atlas_slide_moved, atlas_mask_moved):
    '''
    params:
      static_img: numpy array
      move_img: numpy array
    '''
    data_img_work_slice = data_img_work[int(data_img_work.shape[0]/2), :, :]

    metric = MutualInformationMetric(nbins=50,
                                 sampling_proportion=None)

    affreg =  AffineRegistration(level_iters=[10,100,100],
                            metric=metric,
                            method='BFGS')

    transform = RigidTransform2D()


    affine_rigidTransform = affreg.optimize(data_img_work_slice,
                         atlas_slide_moved,
                         transform,
                         static_mask=atlas_mask_moved.astype(np.float32),
                         params0=None)

    atlas_RigidTransform = affine_rigidTransform.transform(atlas_slide_moved,
                        interpolation='linear')

    return atlas_RigidTransform


  def extract_brain(self, img_nii, img_transform):
    '''
    params:
      img: numpy array
    '''
    data_img_work_slice = img_nii.get_fdata()[self.slide_img, :, :]
    brain = np.multiply(data_img_work_slice, img_transform)
    #
    return brain

  def save_img(self, img, path):
    '''
    params:
      img: numpy array
      path: str
    '''
    pltImage.imsave(f'{self.path_output}{path}.png}', img, cmap='gray')


  def run(self, img):
    '''
    params:
      img: str
    '''
    img_work_nii, img_work_data = self.load_data(img)
    path = img.split('/')[-1].replace('.nii', '')
    print(path, img_work_nii.shape)
    mask_atlas, matched_atlas_resize = self.prepare_data(img_work_nii)
    atlas_slide_moved, atlas_mask_moved = self.center_brain(img_work_data,
                                                            matched_atlas_resize,
                                                            mask_atlas)
    atlas_RigidTransform = self.apply_register(img_work_data,
                                               atlas_slide_moved,
                                               atlas_mask_moved)
    brain = self.extract_brain(img_work_nii, atlas_RigidTransform)
    self.save_img(brain, path)



#Ejecución

> Se pasa al proceso el template del cerebro que se va a utilizar y el directorio con los datos de entrada a tratar



In [ ]:
vibe = ViBE("all_template.nii",'/content/drive/MyDrive/data_nii_salida/')
files_nii = sorted(glob.glob('/content/drive/MyDrive/data_nii_resize_max_18/*'))
for file in files_nii:
  print("Procesando:", file)
  vibe.run(file)